~~~
Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
~~~

# Hugging Face를 활용한 강화학습

<table><tbody><tr>
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/google-health/medgemma/blob/main/notebooks/reinforcement_learning_with_hugging_face.ipynb">
      <img alt="Google Colab logo" src="https://www.tensorflow.org/images/colab_logo_32px.png" width="32px"><br> Google Colab에서 실행
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogle-Health%2Fmedgemma%2Fmain%2Fnotebooks%2Freinforcement_learning_with_hugging_face.ipynb">
      <img alt="Google Cloud Colab Enterprise logo" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br> Colab Enterprise에서 실행
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/google-health/medgemma/blob/main/notebooks/reinforcement_learning_with_hugging_face.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> GitHub에서 보기
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://huggingface.co/collections/google/medgemma-release-680aade845f90bec6a3f60c4">
      <img alt="Hugging Face logo" src="https://huggingface.co/front/assets/huggingface_logo-noborder.svg" width="32px"><br> View on Hugging Face
    </a>
  </td>
</tr></tbody></table>

이 노트북은 `trl`: Transformer Reinforcement Learning을 통해 의료용 QA를 위한 텍스트 데이터세트에서 MedGemma를 RL 튜닝(RL-tuning)하는 것을 보여줍니다. 이 노트북은 MedGemma 1.5의 기본 기능을 보여주기 위한 교육 목적으로만 제공됩니다. 완성되거나 승인된 제품을 나타내지 않으며 어떠한 질병이나 상태도 진단하거나 치료를 제안할 목적이 없으며, 의학적 조언으로 사용되어서는 안 됩니다. 자세한 내용은 HAI-DEF <a href='https://developers.google.com/health-ai-developer-foundations/terms?'>서비스 약관</a>을 참조하십시오.

**동기(Motivation)**: 기본적으로 MedGemma는 미국 의사 면허 시험(USMLE)과 유사한 질문들에 대해 긴 추론(thinking) 과정과 답변을 제공합니다. 효율성을 위해 모델의 출력을 1K(1,000) 개의 토큰으로 제한하려고 합니다.

이 가이드에선 [`trl`](https://huggingface.co/docs/trl/grpo_trainer) - 트랜스포머 강화학습(Transformer Reinforcement Learning)을 이용하여 강화학습(RL, Reinforcement Learning)으로 모델을 학습시킵니다. 높은 성능을 유지하면서 계산 비용을 줄이기 위해 [로우랭크 어댑테이션(LoRA, Low-Rank Adaptation)](https://arxiv.org/abs/2106.09685v2)을 활용하는 [그룹 간의 상대적 정책 최적화(GRPO, Group Relative Policy Optimization)](https://arxiv.org/abs/2402.03300) 기법을 특별적으로 사용합니다.

**인용:**

- LoRA: Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., ... & Chen, W. (2022). Lora: Low-rank adaptation of large language models. ICLR, 1(2), 3.
- LoRA: Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., ... & Chen, W. (2022). Lora: Low-rank adaptation of large language models. ICLR, 1(2), 3.

- GRPO: Shao, Z., Wang, P., Zhu, Q., Xu, R., Song, J., Bi, X., ... & Guo, D. (2024). Deepseekmath: Pushing the limits of mathematical reasoning in open language models. arXiv preprint arXiv:2402.03300.

## 설정

이 튜토리얼을 완료하려면 MedGemma 모델을 파인튜닝할 수 있는 충분한 리소스가 있는 런타임이 필요합니다. **참고:** 이 가이드에는 bfloat16 데이터 유형을 지원하고 메모리가 40GB 이상인 GPU가 필요합니다.

A100 GPU를 사용하여 Google Colab에서 이 노트북을 실행할 수 있습니다:

1. Colab 창 오른쪽 상단에서 **▾ (추가 연결 옵션)**을 선택합니다.
2. **런타임 유형 변경(Change runtime type)**을 선택합니다.
3. **하드웨어 가속기(Hardware accelerator)**에서 **A100 GPU**를 선택합니다.

*이 작업을 실행하는 데 오랜 시간이 걸립니다 (A100 40GB GPU에서 1700개의 학습 단계에 약 11시간 소요).* 


### MedGemma 접근 권한 얻기

시작하기 전에 Hugging Face에서 MedGemma 모델에 대한 접근 권한이 있는지 확인하세요:

1. 아직 Hugging Face 계정이 없다면 [여기](https://huggingface.co/join)를 클릭하여 무료로 계정을 만들 수 있습니다.
2. [MedGemma 모델 페이지](https://huggingface.co/google/medgemma-1.5-4b-it)로 이동하여 사용 조건(usage conditions)에 동의합니다.



### Hugging Face 토큰 구성

[설정(settings)](https://huggingface.co/settings/tokens)으로 이동하여 Hugging Face `write` 액세스 토큰을 생성합니다. **참고:** 파인튜닝된 모델을 Hugging Face Hub에 푸시하려면 토큰에 쓰기(write) 권한이 있어야 합니다.

Google Colab을 사용하는 경우 Colab Secrets 관리자에 액세스 토큰을 추가하여 안전하게 저장하세요. 그렇지 않은 경우 아래 셀을 실행하여 Hugging Face에 인증을 진행합니다.

1. Google Colab 노트북을 열고 왼쪽 패널의 🔑 Secrets 탭을 클릭합니다. <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
1. Google Colab 노트북을 열고 왼쪽 패널의 🔑 Secrets 탭을 클릭합니다. <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. `HF_TOKEN`이라는 이름으로 새 비밀키를 만듭니다.
3. 발급받은 토큰 키를 `HF_TOKEN`의 값(Value) 입력 상자에 복사/붙여넣기 합니다.
4. 왼쪽 버튼을 토글하여 노트북이 비밀키에 액세스할 수 있도록 허용합니다.
5. 원활한 후속 `trl` 실행을 위해 `HF_HOME`도 설정합니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    # Google Colab에서 실행하는 경우 시크릿 사용
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Colab Enterprise에서 실행 중인 경우 Hugging Face 데이터를 `/content` 아래에 저장
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Hugging Face 인증
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [ ]:
!pip install -q -U transformers trl[vllm] datasets tensorflow fastai gensim wandb
# python 3.12 및 !pip install transformers==4.57.3 trl[vllm]==4.4.1 torch==2.8.0+cu128 환경에서 테스트됨


In [ ]:
# Colab에서 실행하는 경우, 학습 과정을 Google Drive에
# 저장하는 것이 좋습니다. (모델 체크포인트를 저장하는 것을 강력히 권장)
if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    from google.colab import drive
    drive.mount('/content/drive')

## 데이터세트 처리

이 노트북은 의학 지식과 임상 추론 능력을 평가하기 위해 고안된 미국, 중국, 대만의 의사 면허 시험에서 파생된 다중 선택형 질문 데이터세트인 [MedQA](https://arxiv.org/abs/2009.13081)를 사용합니다.

Hugging Face `datasets` 라이브러리를 사용하여 데이터를 로드합니다. 그런 다음, 학습(train) 및 검증(validation) 분할을 만듭니다. 우리는 더 빠른 평가 시간을 위해 dev 분할을 서브샘플링합니다.

**데이터세트 인용:** Jin, D., Pan, E., Oufattole, N., Weng, W. H., Fang, H., & Szolovits, P. (2021). What disease does this patient have? a large-scale open domain question answering dataset from medical exams. Applied Sciences, 11(14), 6421.


In [ ]:
import datasets

def process_medqa(data):
    prompt_template = f"""Answer the given question. Think step by step.
    You can directly provide the answer (A single letter), without further additions. E.g. "Final Answer: (A)".
    Question: [QUESTION]
    [OPTIONS]
    """
    return data.map(lambda x: {
                        'prompt': [
                            {'role': 'system', 'content': 'SYSTEM INSTRUCTION: think silently if needed.'},
                            {'role': 'user', 'content': prompt_template.replace('[QUESTION]', x['data']['Question']).replace(
                                '[OPTIONS]', f"(A) {x['data']['Options']['A']} (B) {x['data']['Options']['B']} (C) {x['data']['Options']['C']} (D) {x['data']['Options']['D']}")}
                        ],
                        'answer': x['data']['Correct Option']
                    })

medqa_dataset = datasets.load_dataset("openlifescienceai/medqa")
train_dataset = process_medqa(medqa_dataset["train"])
val_dataset = process_medqa(medqa_dataset["dev"])

In [ ]:
train_dataset['prompt'][0]

[{'content': 'SYSTEM INSTRUCTION: think silently if needed.',
  'role': 'system'},
 {'content': 'Answer the given question. Think step by step.\n  You can directly provide the answer (A single letter), without further additions. E.g. "Final Answer: (A)".\n  Question: A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?\n  (A) Ampicillin (B) Ceftriaxone (C) Doxycycline (D) Nitrofurantoin\n  ',
  'role': 'user'}]

## MedQA에 대한 GRPO를 통한 LoRA 사후 학습(Post-train)

대규모 언어 모델(LLM)의 전통적인 파인튜닝은 수십억 개의 매개변수를 조정해야 하므로 리소스 집약적입니다. 매개변수 효율적 조정(PEFT, Parameter-Efficient Fine-Tuning)은 더 적은 수의 매개변수를 학습하여 이 문제를 해결합니다. 일반적인 PEFT 기법은 *Low-Rank Adaptation (LoRA)*으로, 전체 가중치 매트릭스를 업데이트하는 대신 원래 모델에 추가되는 작고 순위가 낮은(low-rank) 매트릭스를 학습하여 대규모 언어 모델을 효율적으로 적용합니다.

*GRPO (Group Relative Policy Optimization)*는 별도의 가치 함수(value function) 필요성을 제거하여 효율성을 높이고 훈련 비용을 줄이는 것을 목표로 하는 강화 학습(RL) 알고리즘입니다. 대신 GRPO는 그룹 기반 어드밴티지 추정을 사용하고 더 나은 안정성을 위해 손실 함수에 KL 발산을 통합합니다.

이 노트북은 LoRA를 사용하여 MedGemma(검증 가능한 보상 사용)의 강화학습(RL) 튜닝을 보여줍니다.


먼저, 모델의 정답 문자가 올바른 정답 문자인지 (즉, 'A', 'B', 'C', 또는 'D') 확인할 수 있도록 보상 함수(reward function)를 정의합니다.


In [ ]:
import re

def extract_xml_answer(answer: str) -> str:
    """Extract the answer letter from an XML answer string."""
    if not isinstance(answer, str):
        return None
    if not answer:
        return None

    final_answers = [
        r'The final answer is\s\(([A-J])\)',
        r'The final answer is\s\**\(([A-J])\)\**',
        r'The final answer is\s\$\\boxed{([A-J])}\$',
        r'Final Answer:\(([A-J])\)',
        r'Final Answer:\s\(([A-J])\)',
        r'Final Answer:\s\(?([A-J])',
        r'Final Answer:\s*\**\(([A-J])\)\**',
        r'\**Final Answer:\**\s\(([A-J])\)',
    ]
    for final_type in final_answers:
        match = re.search(final_type, answer)
        if match:
            answer_letter = match.group(1)
            return f'{answer_letter}'

    return None

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward function to check when the model's answer letter matches the correct answer letter (i.e. 'A', 'B', ...)"""
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    # print(f"-----질문:\n{q}\n정답:\n{answer[0]}\n응답:\n{responses[0]}\n추출됨:\n{extracted_responses[0]}")
    # print([(r,a, r == a) for r, a in zip(extracted_responses, answer)])
    return [1.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

다음으로, `GRPOConfig`를 사용하여 학습을 구성합니다.


In [ ]:
import torch
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

ckpt = "google/medgemma-1.5-4b-it"
output_dir="./tuned_medgemma4b",

training_args = GRPOConfig(
    output_dir=output_dir,
    eval_on_start=False,                     # Run an evaluation at the very beginning of training.
    learning_rate=5e-6,                      # The initial learning rate for the AdamW optimizer.
    per_device_train_batch_size=3,
    gradient_accumulation_steps=4,           # Accumulate gradients for this many steps to simulate a larger batch size (per_device_train_batch_size * gradient_accumulation_steps).
    num_generations=4,                       # Number of completions to generate per prompt for GRPO's preference learning.
    max_prompt_length=512,                   # Maximum token length for input prompts.
    max_completion_length=1024,              # Maximum token length for the model's generated completions.
    max_steps=1700,
    logging_steps=20,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    report_to="tensorboard",
    use_vllm=True,                           # Use the vLLM library for significantly faster inference during generation.
    vllm_mode="colocate",                    # vLLM deployment mode; 'colocate' runs vLLM on the same GPU(s) as the trainer.
    vllm_gpu_memory_utilization=.30,         # Fraction of GPU memory that vLLM is allowed to use.
    bf16=True,                               # Enable bfloat16 mixed precision training to save memory and speed up training.
    gradient_checkpointing=True,             # Save memory by trading compute (avoids storing all intermediate activations).
    gradient_checkpointing_kwargs={
        "use_reentrant": False               # Use a more efficient implementation of gradient checkpointing.
    },
    model_init_kwargs={
        "device_map": "auto",
        "dtype": torch.bfloat16,             # Set model parameter data type to bfloat16.
        "attn_implementation": "eager"       # Gemma 3 recommends using the 'eager' attention implementation.
    },
    push_to_hub=True
)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=64,
    lora_alpha=64,
    target_modules="all-linear",
)

모델을 학습시킵니다.

이 과정을 실행하는 데에는 오랜 시간이 걸립니다 (A100 40GB GPU에서 총 약 11시간 소요).


In [ ]:
trainer = GRPOTrainer(
    model=ckpt,
    reward_funcs=[correctness_reward_func],
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset.select(range(100)), # Use a very small subset for validation
    peft_config=lora_config,
)
trainer.train()
trainer.save_model(output_dir=training_args.output_dir)

In [ ]:
# 훈련 결과를 Google Drive에 저장하려면 관련 경로를 변경하세요
if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    ! cp -r ./tuned_medgemma4b/ /content/drive/MyDrive/trl_colab_storage/

In [ ]:
# 훈련 루프(curves) 시각화
! pip install -q tensorboard
%load_ext tensorboard
%tensorboard --logdir /content/tuned_medgemma4b/ --port 6007

## 모델 평가: RL-tuning의 효과

**중요: 계속 진행하기 전에, Colab 커널의 VRAM 제한으로 인해 런타임을 다시 시작해야 할 수도 있습니다.**

다음 셀들은 테스트 데이터세트에 대해 베이스라인 모델과 파인튜닝된 모델의 정확도를 계산하고 출력하여 RL 튜닝의 효과를 평가합니다.

우리는 또한 이전과 동일한 로직을 사용하여 테스트 분할을 로드하고 처리합니다. 편의를 위해 이 함수들이 아래에 반복되어 있습니다.


In [ ]:
# 환경 변수 재구성
import os
import sys

if "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT"):
    # Google Colab에서 실행하는 경우 시크릿 사용
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
else:
    # Colab Enterprise에서 실행 중인 경우 Hugging Face 데이터를 `/content` 아래에 저장
    if os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE":
        os.environ["HF_HOME"] = "/content/hf"
    # Hugging Face 인증
    from huggingface_hub import get_token
    if get_token() is None:
        from huggingface_hub import notebook_login
        notebook_login()

In [ ]:
import datasets
import re

def extract_xml_answer(answer: str):
    """Extract the answer letter from an XML answer string."""
    if not isinstance(answer, str):
        return None
    if not answer:
        return None

    final_answers = [
        r'The final answer is\s\(([A-J])\)',
        r'The final answer is\s\**\(([A-J])\)\**',
        r'The final answer is\s\$\\boxed{([A-J])}\$',
        r'Final Answer:\(([A-J])\)',
        r'Final Answer:\s\(([A-J])\)',
        r'Final Answer:\s\(?([A-J])',
        r'Final Answer:\s*\**\(([A-J])\)\**',
        r'\**Final Answer:\**\s\(([A-J])\)',
    ]
    for final_type in final_answers:
        match = re.search(final_type, answer)
        if match:
            answer_letter = match.group(1)
            return f'{answer_letter}'

    return None


def process_medqa(data):
    prompt_template = f"""Answer the given question. Think step by step.
    You can directly provide the answer (A single letter), without further additions. E.g. "Final Answer: (A)".
    Question: [QUESTION]
    [OPTIONS]
    """
    return data.map(lambda x: {
                        'prompt': [
                            {'role': 'system', 'content': 'SYSTEM INSTRUCTION: think silently if needed.'},
                            {'role': 'user', 'content': prompt_template.replace('[QUESTION]', x['data']['Question']).replace(
                                '[OPTIONS]', f"(A) {x['data']['Options']['A']} (B) {x['data']['Options']['B']} (C) {x['data']['Options']['C']} (D) {x['data']['Options']['D']}")}
                        ],
                        'answer': x['data']['Correct Option']
                    })

medqa_dataset = datasets.load_dataset("openlifescienceai/medqa")
test_dataset = process_medqa(medqa_dataset["test"])

테스트 데이터 세트에 대해 배치 추론을 실행하는 메소드를 정의합니다.


In [ ]:
import torch
import pandas as pd
from tqdm.auto import tqdm

def run_inference_batched(test_dataset, model, processor, batch_size=4, device="cuda", verbose=True):
    """
    Runs inference on a processed test dataset using batching for efficiency.

    Args:
        test_dataset: A dataset where each item has 'prompt' (chat history) and 'answer' (ground truth).
        model: The loaded PEFT model for inference.
        processor: The processor for tokenizing the input.
        batch_size (int): The number of samples to process at once. Adjust based on VRAM.
        device (str): The device to run inference on ('cuda' or 'cpu').
        verbose (bool): Whether to print progress and sample outputs.

    Returns:
        A list of dictionaries, with each dictionary containing the prompt,
        ground truth answer, and the model's generated answer.
    """
    results = []

    # 배치를 위한 반복자 생성
    num_samples = len(test_dataset)

    # verbose가 True인 경우 진행 표시줄을 위해 tqdm 사용
    batch_iterator = range(0, num_samples, batch_size)
    if verbose:
        print(f"Starting batched inference on {num_samples} samples with batch size {batch_size}...")
        batch_iterator = tqdm(batch_iterator, desc="Batch Inference")

    for i in batch_iterator:
        # 1. 현재 배치 준비
        batch_data = test_dataset[i : i + batch_size]
        batch_prompts = batch_data['prompt']
        batch_ground_truths = batch_data['answer']

        # 2. 왼쪽 패딩으로 전체 배치를 한 번에 토큰화
        inputs = processor.tokenizer.apply_chat_template(
            batch_prompts,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            truncation=True,
            padding=True,  # Pad sequences to the length of the longest in the batch
            max_length=1024,  # Set a fixed maximum length
        ).to(device)

        # 3. 전체 배치에 대한 응답을 한 번에 생성
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False, # Greedy decoding for deterministic output appropriate for logical reasoning"
        )

        # 4. 출력의 생성된 부분만 디코딩
        # 이것은 전체 시퀀스를 디코딩하고 프롬프트를 제거하는 것보다 더 견고합니다
        input_token_length = inputs['input_ids'].shape[1]
        generated_tokens = outputs[:, input_token_length:]
        model_generated_answers = processor.batch_decode(generated_tokens, skip_special_tokens=True)

        # 5. 현재 배치에 대한 결과 저장
        for j in range(len(batch_prompts)):
            results.append({
                'prompt': batch_prompts[j],
                'ground_truth': batch_ground_truths[j],
                'model_answer': model_generated_answers[j].strip() # Use .strip() for clean output
            })

    # 선택 사항: 최종 결과의 몇 가지 예시 출력
    if verbose:
        print("\n--- Sample of Batched Inference Results ---")
        for res in results[:3]: # Print first 3 results
            print(f"Ground Truth: {res['ground_truth']}")
            print(f"Model Answer: {res['model_answer']}\n")

    return results

### 베이스라인 성능 평가

이 셀은 테스트 데이터에 대한 베이스라인 모델의 정확도를 계산합니다. 이 베이스라인은 파인튜닝된 모델의 성능 향상을 측정하기 위한 벤치마크 역할을 합니다.


In [ ]:
from transformers import Gemma3Processor, AutoModelForCausalLM

# 모델 및 프로세서 로드
print("Loading model and processor...")
ckpt = "google/medgemma-1.5-4b-it"
model = AutoModelForCausalLM.from_pretrained(
    ckpt,
    dtype=torch.bfloat16,
    device_map="auto",
)
processor = Gemma3Processor.from_pretrained(ckpt)

# 테스트 데이터세트에 대한 추론 실행
inference_results = run_inference_batched(
    test_dataset=test_dataset,
    model=model,
    processor=processor,
    batch_size=32,
)
results_df = pd.DataFrame(inference_results)
results_df['model_pred'] = results_df['model_answer'].apply(extract_xml_answer)
results_df['correct'] = results_df['ground_truth'] == results_df['model_pred']
print('Baseline Accuracy', results_df['correct'].mean())
results_df.to_csv('baseline_test_results.csv') # Save baseline results

del model # To free up VRAM
del processor
torch.cuda.empty_cache()

Baseline Accuracy 0.14072327044025157


예상대로, 1k 출력 토큰만으로는 14%의 낮은 정확도를 관찰할 수 있습니다. 결과 예시를 살펴보겠습니다.


In [ ]:
print(results_df['model_answer'].values[0])

<unused94>thought
The user wants me to identify the antibiotic that blocks cell wall synthesis based on the clinical presentation and lab findings.

1.  **Analyze the clinical presentation:** A young, sexually active male presents with symptoms suggestive of urethritis (fever, dysuria) and septic arthritis (knee pain, inflammation). This points towards a potential sexually transmitted infection (STI) that can cause disseminated infection, like *Neisseria gonorrhoeae*.
2.  **Analyze the lab findings:** The joint fluid culture shows a bacterium that does not ferment maltose and has no polysaccharide capsule.
    *   *Neisseria gonorrhoeae* is the most common cause of septic arthritis in young, sexually active adults.
    *   *Neisseria gonorrhoeae* typically ferments glucose but *not* maltose.
    *   *Neisseria gonorrhoeae* typically has a polysaccharide capsule, although some strains may lack it (non-encapsulated strains).
    *   However, the description "does not ferment maltose and 

이 경우(그리고 다른 많은 경우) 1k 출력 제한은 너무 낮아서 모델이 최종 답변으로 응답하기 전에 모델의 응답이 잘려 버립니다.


### 튜닝된 모델 성능 평가

이 셀은 테스트 데이터에 대한 파인튜닝된 모델의 정확도를 계산합니다. 이 점수를 베이스라인 점수와 비교하면 파인튜닝을 통해 얻은 개선 사항을 알 수 있습니다.


In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import Gemma3Processor
import torch
import pandas as pd

# 모델 및 프로세서 정보 정의 (경로가 올바른지 확인하세요!)
model_path = "/content/tuned_medgemma4b/checkpoint-1700"

# 모델 및 프로세서 로드
print("Loading model and processor...")
model = AutoPeftModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map="auto",
)
processor = Gemma3Processor.from_pretrained("google/medgemma-1.5-4b-it")

# 처리된 전체 데이터세트에 대한 추론 실행
inference_results = run_inference_batched(
    test_dataset=test_dataset,
    model=model,
    processor=processor,
    batch_size=64,
)
results_df = pd.DataFrame(inference_results)
results_df['model_pred'] = results_df['model_answer'].apply(extract_xml_answer)
results_df['correct'] = results_df['ground_truth'] == results_df['model_pred']
print('GRPO-tuned Accuracy', results_df['correct'].mean())
results_df.to_csv('trained_results.csv') # Save trained_results

GRPO-tuned Accuracy 0.7051886792452831


1700 단계 후 MedQA **1k 출력** 테스트 정확도 재현:

---



| 모델 | RL 튜닝 이전 | RL 튜닝 이후 |
| -------- | ------- | ------- |
| medgemma-1.5-4b-it | 0.141 | 0.705 |

관찰 사항:
- 정확도 회복: GRPO를 통한 RL 튜닝은 베이스라인 14.1%에서 70.5%로 모델 정확도를 향상시켰습니다.
- 토큰 제한 문제 수정: 베이스라인 모델은 긴 추론 체인을 생성하여 최종 답을 제시하기 전에 1,000 토큰 제한에 걸려 잘리는 경우가 종종 발생했습니다. 파인튜닝된 모델은 더 간결해지고 제한 내에서 답을 제공하도록 학습되었습니다.
- 높은 효율성: 비교적 작은 4B 파라미터 모델로 MedQA 데이터세트에서 이 정도 수준의 성능(70.5% 정확도)을 달성한 참조를 포함하여 도메인 적응을 위해 LoRA와 GRPO를 사용하는 것의 효과를 입증합니다.



### 추가적인 최적화
이 노트북은 출발점일 뿐이라는 점을 알아두세요. [deepspeed](https://huggingface.co/docs/trl/main/en/deepspeed_integration), [다중 노드에서의 병렬화](https://huggingface.co/docs/trl/main/en/grpo_trainer#grpo-at-scale-train-a-70b-model-on-multiple-nodes) 등을 포함하여 이 Colab에서 다루지 않은 수많은 최적화가 있습니다.

자세한 내용은 [GRPO Trainer](https://huggingface.co/docs/trl/main/en/grpo_trainer)를 참조하시기 바랍니다.


## 다음 단계

다른 [노트북들](https://github.com/google-health/medgemma/blob/main/notebooks)을 살펴보고 이 모델로 어떤 것들을 더 할 수 있는지 알아보세요.
